# Sentiment Analysis Using Bag of Words and Logistic Regression

## Project Overview

Natural Language Processing (NLP) enables machines to understand, analyze, and extract meaningful information from human language. One of the most fundamental NLP tasks is **Sentiment Analysis**, which aims to determine whether a piece of text expresses a positive or negative opinion.

In this project, I build a complete sentiment analysis pipeline using classical NLP techniques and Machine Learning algorithms. The workflow covers the entire process from raw text preprocessing to feature extraction, model training, evaluation, and experiment documentation.

The primary objective of this project is not only to achieve high predictive performance but also to systematically investigate how different preprocessing techniques, feature engineering strategies, and machine learning models affect sentiment classification performance.

---

## Dataset

Dataset Link: **[IMBD Movie Reviews](https://www.kaggle.com/datasets/mwallerphunware/imbd-movie-reviews-for-binary-sentiment-analysis)**

The dataset consists of movie reviews labeled with their corresponding sentiment:

* Positive
* Negative

The reviews contain real-world natural language, making them suitable for evaluating text preprocessing techniques and machine learning models.

---

## Project Pipeline

The following stages are implemented throughout this project:

### 1. Text Preprocessing

* Contraction expansion
* Lowercasing
* Text cleaning using regular expressions
* Stopword removal with preserved negations
* Lemmatization
* Corpus construction

### 2. Feature Extraction

* Bag of Words (CountVectorizer)
* N-gram generation
* Vocabulary analysis
* Feature selection using `max_features`
* Rare-word filtering using `min_df`

### 3. Model Training

Different machine learning algorithms are evaluated and compared, including:

* Gaussian Naive Bayes
* Multinomial Naive Bayes
* Bernoulli Naive Bayes
* Logistic Regression
* Support Vector Machines (SVM)
* Decision Trees
* Random Forests

### 4. Model Evaluation

Performance is assessed using multiple metrics:

* Accuracy
* Precision
* Recall
* F1-Score
* ROC-AUC Score
* Confusion Matrix
* Cross-Validation Mean Accuracy
* Cross-Validation Standard Deviation

---

## Experimental Approach

Rather than training a single model, this notebook follows an experimentation-driven methodology.

For each model, multiple configurations are tested, including different values for:

* `max_features`
* `min_df`
* N-gram ranges
* Preprocessing strategies

The goal is to identify the most effective configuration while understanding the trade-offs between model complexity, computational cost, and predictive performance.

---

## Key Learning Objectives

Through this project, I aim to:

* Develop a deeper understanding of NLP preprocessing techniques.
* Compare the behavior of different machine learning algorithms on text data.
* Analyze how feature engineering impacts classification performance.
* Build reproducible NLP pipelines suitable for real-world applications.
* Establish strong baselines before moving toward advanced embedding and transformer-based approaches.

---

**Author:** Hazem Mohamed

**Role:** AI Engineer | Machine Learning Engineer | NLP Engineer

**Repository:** [NLP Experimentation Lab](https://github.com/Hazem1695/NLP-Experimentation-Lab)


# **Importing the Libraries**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# **Data preprocessing**

## Data Cleaning Check Template
This template is designed to quickly assess the quality of any dataset before building machine learning models or performing analysis.

It provides a structured overview of the dataset by checking for common data issues such as:

- Missing values

- Duplicate rows

- Incorrect data types

- Outliers

- Distribution of numerical features

- Categorical feature consistency

**What This Template Does**

- Displays basic dataset information (shape, data types)

- Identifies missing values and duplicates

- Summarizes numerical and categorical features

- Detects potential outliers using the IQR method

- Highlights columns with low unique values for quick inspection

How to Use

1. Load your dataset using Pandas  

2. Call the function:

In [2]:
def data_quality_report(df):

    print("DATA QUALITY REPORT")
    
    # Print a separator line for better readability
    
    print("=" * 50)
    print("BASIC INFO")
    print("=" * 50)
    
    # Show general information about the dataset (columns, data types, non-null values)
    print(df.info())
    
    # Show number of rows and columns
    print("\n" + "=" * 50)
    print("SHAPE OF DATA")
    print("=" * 50)
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    
    # Check for missing (null) values in each column
    print("\n" + "=" * 50)
    print("MISSING VALUES")
    print("=" * 50)
    missing = df.isnull().sum()
    
    # Display only columns that have missing values
    print(missing[missing > 0])
    
    # Check for duplicate rows
    print("\n" + "=" * 50)
    print("DUPLICATES")
    print("=" * 50)
    print(f"Duplicate rows: {df.duplicated().sum()}")
    
    # Display data types of each column
    print("\n" + "=" * 50)
    print("DATA TYPES")
    print("=" * 50)
    print(df.dtypes)
    
    # Summary statistics for numerical columns (mean, std, min, max, etc.)
    print("\n" + "=" * 50)
    print("NUMERICAL SUMMARY")
    print("=" * 50)
    print(df.describe())
    
    # Summary for categorical (object) columns
    print("\n" + "=" * 50)
    print("CATEGORICAL SUMMARY")
    print("=" * 50)
    print(df.describe(include=['object']))
    
    # Show unique values for columns with low number of distinct values
    # Useful for detecting categories, errors, or inconsistencies
    print("\n" + "=" * 50)
    print("UNIQUE VALUES (LOW CARDINALITY)")
    print("=" * 50)
    for col in df.columns:
        if df[col].nunique() < 10:  # Only show columns with few unique values
            print(f"{col}: {df[col].unique()}")
            
    # correlation
    print("\n" + "=" * 50)
    print("CORRELATION MATRIX")
    print("=" * 50)
    print(df.corr(numeric_only=True))
    
    # Detect outliers using the IQR (Interquartile Range) method
    print("\n" + "=" * 50)
    print("OUTLIERS CHECK (IQR METHOD)")
    print("=" * 50)
    
    # Loop through only numerical columns
    for col in df.select_dtypes(include=np.number).columns:
        Q1 = df[col].quantile(0.25)  # 25th percentile
        Q3 = df[col].quantile(0.75)  # 75th percentile
        IQR = Q3 - Q1  # Interquartile range
        
        # Count rows that fall outside the normal range
        outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
        print(f"{col}: {len(outliers)} outliers")

## **Load dataset**
Apply Data Cleaning Check Template

In [ ]:
dataset = pd.read_csv('MovieReviewTrainingDatabase.csv')
data_quality_report(dataset)

DATA QUALITY REPORT
BASIC INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  25000 non-null  object
 1   review     25000 non-null  object
dtypes: object(2)
memory usage: 390.8+ KB
None

SHAPE OF DATA
Rows: 25000, Columns: 2

MISSING VALUES
Series([], dtype: int64)

DUPLICATES
Duplicate rows: 96

DATA TYPES
sentiment    object
review       object
dtype: object

NUMERICAL SUMMARY
       sentiment                                             review
count      25000                                              25000
unique         2                                              24904
top     Positive  You do realize that you've been watching the E...
freq       12500                                                  3

CATEGORICAL SUMMARY
       sentiment                                             review
count      25000                 

## Duplicate Data Detection

In [4]:
duplicates = dataset[dataset.duplicated(subset=['review'], keep=False)]
duplicates.sort_values('review')

,sentiment,review
21186,Negative,"Back in his youth, the old man had wanted to..."
21877,Negative,"Back in his youth, the old man had wanted to..."
14734,Negative,'Dead Letter Office' is a low-budget film abou...
5519,Negative,'Dead Letter Office' is a low-budget film abou...
7011,Positive,".......Playing Kaddiddlehopper, Col San Fernan..."
...,...,...
2685,Negative,"in this movie, joe pesci slams dunks a basketb..."
22244,Positive,it's amazing that so many people that i know h...
14767,Positive,it's amazing that so many people that i know h...
12462,Negative,this movie begins with an ordinary funeral... ...


## Quantifying Duplicate Review Frequencies

In [5]:
review_counts = dataset['review'].value_counts()
print("Reviews appearing more than once:")
print((review_counts > 1).sum())
print("\nMaximum repetitions:")
print(review_counts.max())

Reviews appearing more than once:
92

Maximum repetitions:
3


## Removing Duplicate Reviews & Resetting Index
> **Note:** This cell drops the repeated rows we identified in the previous steps and cleanly resets the row indices for model training

In [6]:
print("Before:", len(dataset))
dataset = dataset.drop_duplicates()
print("After:", len(dataset))
dataset = dataset.reset_index(drop=True)

Before: 25000
After: 24904


## Library Installation
> **Note:** The `contractions` library is required to automatically expand shortcuts like *don't* to *do not* and *I'm* to *I am* during preprocessing.

In [7]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 7.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.8 MB/s eta 0:00:00


## **Cleaning the texts**

In [8]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
import re
import contractions
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

base_stopwords = set(stopwords.words('english'))
negation_words = {'not', 'no', 'never'}
all_stopwords = base_stopwords - negation_words

wnl = WordNetLemmatizer()

corpus = []

for i in range(0, len(dataset)):
    review = dataset['review'][i]
    # Fix contractions
    review = contractions.fix(review)
    # Lowercase
    review = review.lower()
    
    review = re.sub(r'[^a-zA-Z\s]', ' ', review)
    # Split
    words = review.split()
    # Chained Lemmatization (Handles both Verbs 'v' and Nouns 'n')
    review = [wnl.lemmatize(wnl.lemmatize(word, pos='v'), pos='n') for word in words if word not in all_stopwords]
    review = ' '.join(review) 
    corpus.append(review)

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


## Preprocessing Verification
> **Note:** Pulling the first two rows directly as a memory array to confirm that our lowercasing, stopword stripping, and lemmatization pipeline worked correctly before feeding it into the vectorizer.

In [9]:
# Pull the data directly as a fast memory array
raw_samples = dataset['review'].head(2).values

for i in range(2):
    print(f"=== REVIEW #{i+1} ===")
    print(f"RAW:     {raw_samples[i]}\n") 
    print(f"CLEANED: {corpus[i]}")
    print("-" * 50)

=== REVIEW #1 ===
RAW:     With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.  Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.  The actual feature film bit when it final

# **Creating the Bag of Word model**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
# Add ngram_range=(1, 2) so it automatically catches phrases like "not good" or "no clue"
cv = CountVectorizer(max_features=30000, min_df=2, ngram_range=(1, 2))
X = cv.fit_transform(corpus).toarray()
y = dataset.iloc[:, 0].values

In [11]:
len(X[0])

30000

# **Encoding Categorical data Using Label Encoding**


In [12]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

In [14]:
print(y)

[1 1 0 ... 0 0 1]


# Class Balance Check
> **Note:** Using NumPy to verify if our dataset is perfectly balanced between positive and negative reviews before splitting it into training and testing sets.

In [13]:
import numpy as np
# This returns the unique classes and how many times they appear
classes, counts = np.unique(y, return_counts=True)
for c, count in zip(classes, counts):
    print(f"Class {c} contains {count}")

Class 0 contains 12432
Class 1 contains 12472


# **Splitting the dataset into the Training set and Test set**

In [15]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 0)

# **Training the Logistic Regression model on the Training set**

In [16]:
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression(max_iter=1000, random_state = 0)
classifier.fit(X_train,y_train)

LogisticRegression(max_iter=1000, random_state=0)

# **Predicting the Test set results**

In [17]:
y_pred = classifier.predict(X_test)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

[[1 1]
 [1 1]
 [0 0]
 ...
 [0 0]
 [0 0]
 [0 0]]


# **Evaluating the Model Performance**

In [18]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_pred))


accuracies = cross_val_score(estimator=classifier, X=X_train, y=y_train, cv=3)

print("\nMean Accuracy:")
print(accuracies.mean())

print("\nStandard Deviation:")
print(accuracies.std())

Confusion Matrix:
[[2200  320]
 [ 232 2229]]

Accuracy Score:
0.8891788797430235

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.87      0.89      2520
           1       0.87      0.91      0.89      2461

    accuracy                           0.89      4981
   macro avg       0.89      0.89      0.89      4981
weighted avg       0.89      0.89      0.89      4981


ROC-AUC Score:
0.8893726256586882

Mean Accuracy:
0.8806404657933044

Standard Deviation:
0.0030547857360243135


# Logistic Regression Performance Analysis for Text Classification

# 1. Objective

The purpose of this experiment was to evaluate the effectiveness of **Logistic Regression** for binary text classification using a **Bag-of-Words (BoW)** representation generated by **CountVectorizer**. Multiple feature extraction configurations were investigated to determine the impact of vocabulary size and feature engineering on classification performance.

The experiments focused on answering the following research questions:

* How does increasing the vocabulary size affect Logistic Regression?
* Does adding bigram features improve classification performance?
* What is the optimal vocabulary size before performance begins to saturate?
* How stable is the model across multiple cross-validation folds?

---

# 2. Experimental Setup

## Dataset

* Original dataset: **57,415 text documents**
* Duplicate samples removed before training.
* Final evaluation performed on **4,981 testing samples**.

---

## Text Representation

The textual data was converted into numerical vectors using **CountVectorizer**.

The following configurations were evaluated:

| Configuration | Parameters                                        |
| ------------- | ------------------------------------------------- |
| C1            | CountVectorizer()                                 |
| C2            | max_features = 5000, ngram_range=(1,2)            |
| C3            | max_features = 10000, ngram_range=(1,2)           |
| C4            | max_features = 15000, ngram_range=(1,2)           |
| C5            | max_features = 20000, min_df=2, ngram_range=(1,2) |
| C6            | max_features = 25000, min_df=2, ngram_range=(1,2) |
| C7            | max_features = 30000, min_df=2, ngram_range=(1,2) |

The use of **bigrams** allows the model to capture contextual word relationships instead of relying solely on individual words.

---

## Logistic Regression Configuration

```python
LogisticRegression(
    random_state=0,
    max_iter=1000
)
```

The `max_iter` parameter was increased from the default value because the optimizer initially failed to converge on the high-dimensional sparse feature space generated by CountVectorizer.

---

# 3. Initial Observation

When using the default configuration:

```python
CountVectorizer()
```

the execution kernel terminated because the resulting vocabulary became extremely large, leading to excessive memory consumption.

To address this issue, the vocabulary size was limited using the `max_features` parameter while simultaneously introducing bigram features to preserve contextual information.

---

# 4. Convergence Analysis

During the first experiment, Logistic Regression produced the following warning:

> *lbfgs failed to converge because the maximum number of iterations was reached.*

This warning indicates that the optimization algorithm had **not yet reached the optimal solution**, rather than indicating a problem with the model itself.

Increasing

```python
max_iter = 1000
```

allowed the optimizer to converge properly while producing more reliable model coefficients.

Interestingly, the predictive performance remained almost identical before and after increasing the iteration limit, suggesting that the initial solution was already close to the optimum.

---

# 5. Experimental Results

| Vocabulary Size | Accuracy   | ROC-AUC    | CV Mean Accuracy | CV Std     |
| --------------- | ---------- | ---------- | ---------------- | ---------- |
| 5,000           | 85.81%     | 0.8582     | 85.49%           | 0.0026     |
| 10,000          | 87.07%     | 0.8709     | 86.81%           | 0.0033     |
| 15,000          | 87.91%     | 0.8793     | 87.41%           | 0.0046     |
| 20,000          | 88.30%     | 0.8832     | 87.68%           | 0.0037     |
| 25,000          | 88.48%     | 0.8849     | 87.95%           | 0.0028     |
| **30,000**      | **88.92%** | **0.8894** | **88.06%**       | **0.0031** |

---

# 6. Performance Analysis

## Effect of Vocabulary Size

A clear performance trend was observed throughout the experiments.

Increasing the vocabulary size consistently improved every evaluation metric.

The improvement can be attributed to two important factors:

* A larger vocabulary captures a richer representation of the documents.
* Bigram features preserve local word context that cannot be represented by individual words alone.

Performance improvements gradually diminished after approximately **20,000 features**, indicating that the model had already captured most of the discriminative information available in the dataset.

This behavior suggests diminishing returns, where adding additional vocabulary contributes smaller but still measurable gains.

---

## Accuracy Analysis

Model accuracy increased steadily across all experiments.

| Features | Accuracy   |
| -------- | ---------- |
| 5K       | 85.81%     |
| 10K      | 87.07%     |
| 15K      | 87.91%     |
| 20K      | 88.30%     |
| 25K      | 88.48%     |
| **30K**  | **88.92%** |

Overall, the model achieved an improvement of more than **3 percentage points** compared to the smallest vocabulary configuration.

---

## Precision and Recall Analysis

Both classes exhibited highly balanced precision and recall values throughout all experiments.

For the best-performing configuration:

Class 0

* Precision = 0.90
* Recall = 0.87

Class 1

* Precision = 0.87
* Recall = 0.91

The balanced precision-recall trade-off demonstrates that the classifier does not significantly favor one class over the other, reducing both false positives and false negatives.

---

## ROC-AUC Analysis

ROC-AUC increased continuously as more features were introduced.

| Features | ROC-AUC    |
| -------- | ---------- |
| 5K       | 0.8582     |
| 10K      | 0.8709     |
| 15K      | 0.8793     |
| 20K      | 0.8832     |
| 25K      | 0.8849     |
| **30K**  | **0.8894** |

An ROC-AUC close to **0.89** indicates that the classifier has a strong ability to distinguish between the two target classes.

---

## Cross-Validation Analysis

Cross-validation results demonstrate that the model generalizes well.

The highest-performing configuration achieved

* Mean Accuracy = **88.06%**
* Standard Deviation = **0.0031**

The extremely small standard deviation indicates that the model performs consistently across different training and validation splits, suggesting excellent robustness and low variance.

---

# 7. Best Model Configuration

The optimal configuration obtained during the experiments was:

```python
CountVectorizer(
    max_features=30000,
    min_df=2,
    ngram_range=(1,2)
)

LogisticRegression(
    random_state=0,
    max_iter=1000
)
```

Performance:

* Accuracy = **88.92%**
* Precision = **0.89**
* Recall = **0.89**
* F1-score = **0.89**
* ROC-AUC = **0.8894**
* Cross-Validation Accuracy = **88.06%**
* Cross-Validation Standard Deviation = **0.0031**

---

# 8. Key Findings

The experiments reveal several important insights:

* Logistic Regression is highly effective for sparse high-dimensional text classification.
* Increasing the vocabulary size consistently improves predictive performance.
* Incorporating bigrams significantly enhances contextual understanding.
* The use of `min_df=2` successfully removes noisy rare words while preserving meaningful information.
* Model performance begins to saturate beyond approximately **20,000 features**, indicating diminishing returns from additional vocabulary.
* Cross-validation demonstrates excellent stability, confirming that the model generalizes well and is not overfitting.

---

# 9. Comparison with Previous Models

| Model                   | Best Accuracy | ROC-AUC    |
| ----------------------- | ------------- | ---------- |
| Gaussian Naive Bayes    | 81.70%        | 0.8156     |
| Multinomial Naive Bayes | 87.56%        | 0.8755     |
| Bernoulli Naive Bayes   | 87.66%        | 0.8767     |
| **Logistic Regression** | **88.92%**    | **0.8894** |

Compared with all previously evaluated Naive Bayes variants, Logistic Regression achieved the highest Accuracy, ROC-AUC, and cross-validation performance. The discriminative nature of Logistic Regression allows it to learn optimal decision boundaries directly from high-dimensional sparse text features, making it more expressive than the probabilistic assumptions employed by Naive Bayes classifiers.

---

# 10. Final Conclusion

This experimental study demonstrates that Logistic Regression is the strongest-performing classical machine learning algorithm evaluated for this text classification task.

The combination of **CountVectorizer**, **30,000 vocabulary features**, **bigram representation**, and **Logistic Regression** achieved the highest predictive performance while maintaining excellent generalization across cross-validation folds.

The final model reached an accuracy of **88.92%**, an ROC-AUC score of **0.8894**, and a cross-validation accuracy of **88.06%** with a very low standard deviation (**0.0031**), confirming that the model is both accurate and robust.

These findings indicate that Logistic Regression effectively exploits sparse textual features and provides a strong baseline for binary sentiment/text classification tasks. While more advanced deep learning approaches such as Transformers may further improve performance, Logistic Regression offers an excellent balance between predictive accuracy, computational efficiency, interpretability, and scalability, making it a practical choice for real-world text classification applications.
